# TP — Théorie des valeurs extrêmes et classification dans les régions extrêmes

Ce TP reprend les notations du cours *Introduction à la théorie des valeurs extrêmes — Application à la classification*.

Les objectifs sont les suivants.

Les données simulées sont préparées par une cellule fournie. Aucune simulation n'est demandée. Il sera néanmoins nécéssaire de télécharger les données utile pour l'application, le lien est plus loin

## Préparation

Les bibliothèques principales sont `numpy`, `pandas`, `matplotlib` et `scikit-learn`.

Le classifieur linéaire utilisé est `LinearSVC`. La forêt aléatoire utilisée est `RandomForestClassifier`.

La cellule suivante peut être exécutée sans modification.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

GRAINE = 20261005

### Données simulées du TP

Les variables `Z1` et `Z2` sont positives et déjà placées sur une échelle commune. L'étiquette de classe est stockée dans `Y`.

Le rayon possède deux régimes. Le corps de la distribution est borné. Au-delà d'un seuil, la queue est exactement de type Pareto avec un indice $\gamma=1/2.5=0.4$. Cette construction permet d'obtenir une zone de stabilité bien visible sur le graphe de Hill.

Les classes ont volontairement été rendues assez mélangées. Dans le corps, les distributions angulaires des deux classes se recouvrent fortement. Dans la queue, chaque classe possède deux directions privilégiées et ces directions sont partiellement entremêlées. La classification extrême reste donc possible, mais elle n'est ni parfaite ni linéaire.

Le simulateur est fourni. Il n'est pas demandé de le modifier.

In [ ]:
def _preparer_donnees(n=24000, graine=GRAINE):
    rng = np.random.default_rng(graine)

    y = rng.binomial(1, 0.5, size=n)

    proportion_queue = 0.025
    seuil_pareto = 4.0
    alpha = 2.5

    est_dans_queue = rng.random(n) < proportion_queue

    r = np.empty(n)

    corps = ~est_dans_queue
    r[corps] = (
        1
        + (seuil_pareto - 1)
        * rng.beta(2, 5, size=corps.sum())
    )

    r[est_dans_queue] = (
        seuil_pareto
        * (1 + rng.pareto(alpha, size=est_dans_queue.sum()))
    )

    theta1 = np.empty(n)

    concentration_corps = 4

    masque = corps & (y == 0)
    theta1[masque] = rng.beta(
        0.60 * concentration_corps,
        0.40 * concentration_corps,
        size=masque.sum(),
    )

    masque = corps & (y == 1)
    theta1[masque] = rng.beta(
        0.40 * concentration_corps,
        0.60 * concentration_corps,
        size=masque.sum(),
    )

    concentration_queue = 22

    masque = est_dans_queue & (y == 0)
    choix_premier_mode = rng.random(masque.sum()) < 0.50
    centres = np.where(
        choix_premier_mode,
        0.18,
        0.62,
    )
    theta1[masque] = rng.beta(
        centres * concentration_queue,
        (1 - centres) * concentration_queue,
    )

    masque = est_dans_queue & (y == 1)
    choix_premier_mode = rng.random(masque.sum()) < 0.50
    centres = np.where(
        choix_premier_mode,
        0.38,
        0.82,
    )
    theta1[masque] = rng.beta(
        centres * concentration_queue,
        (1 - centres) * concentration_queue,
    )

    z1 = r * theta1
    z2 = r * (1 - theta1)

    return pd.DataFrame(
        {
            "Z1": z1,
            "Z2": z2,
            "Y": y,
        }
    )


donnees = _preparer_donnees()

quantile_rayon_reference = (
    4.0
    * (0.025 / 1e-5) ** (1 / 2.5)
)

donnees.head()

# Partie 1 — Rayon, angle et estimation de la queue

## Rappel de cours

Dans l'orthant positif, le cours utilise

$$
R=\lVert Z\rVert_1=Z_1+Z_2
$$

et

$$
\Theta=\frac{Z}{R}.
$$

En dimension 2, il suffit de conserver

$$
\Theta_1=\frac{Z_1}{R},
\qquad
\Theta_2=1-\Theta_1.
$$

Le rayon mesure la taille de l'événement. L'angle décrit sa direction.

### Question 1

Créer les colonnes `R` et `Theta1` dans `donnees`, puis afficher les cinq premières lignes.

In [ ]:
donnees["R"] = ...  ## A completer ##
donnees["Theta1"] = ...  ## A completer ##

donnees.head()

### Question 2

Tracer le nuage `(Z1, Z2)` en distinguant les deux valeurs de `Y`, en limitant l'affichage à `R < 10`, puis commenter le degré de recouvrement entre les classes et la façon dont la géométrie change lorsque le rayon devient grand.

In [ ]:
masque_affichage = donnees["R"] < 10

plt.figure(figsize=(7, 6))

for classe, marqueur in [(0, "o"), (1, "x")]:
    masque = ...  ## A completer ##

    plt.scatter(
        ...,  ## A completer ##
        ...,  ## A completer ##
        s=12,
        alpha=0.35,
        marker=marqueur,
        label=f"classe {classe}",
    )

plt.xlabel("$Z_1$")
plt.ylabel("$Z_2$")
plt.title("Géométrie des données simulées")
plt.legend()
plt.show()

## Rappel de cours — Statistiques d'ordre et estimateur de Hill

Pour un échantillon positif ordonné

$$
X_{1:n}\leq\cdots\leq X_{n:n},
$$

le seuil associé aux $k$ plus grandes observations est

$$
u=X_{n-k:n}.
$$

L'estimateur de Hill est

$$
\widehat\gamma_k=
\frac{1}{k}
\sum_{i=1}^{k}
\log\left(
\frac{X_{n-k+i:n}}{X_{n-k:n}}
\right).
$$

Un petit $k$ vise une zone très extrême mais donne une forte variance. Un grand $k$ réduit la variance mais peut faire entrer des observations qui appartiennent encore au corps de la distribution.

Ici, la queue de Pareto commence après un rayon égal à 4. Le graphe de Hill devrait donc présenter une zone relativement stable tant que les $k$ observations retenues restent principalement dans cette queue.

### Question 3

Compléter la fonction `estimateur_hill`, puis vérifier qu'elle renvoie une valeur finie pour `k=400`.

In [ ]:
def estimateur_hill(rayons, k):
    rayons_ordonnes = ...  ## A completer ##
    seuil = ...  ## A completer ##
    plus_grands = ...  ## A completer ##
    gamma_chapeau = ...  ## A completer ##
    return gamma_chapeau


estimateur_hill(
    donnees["R"].to_numpy(),
    400,
)

### Question 4

Tracer le graphe de Hill pour `k` allant de 50 à 1200. Repérer une zone où `gamma_chapeau` reste approximativement stable, puis fixer `k_hill` dans cette zone. Expliquer pourquoi la rupture du plateau indique que des observations du corps commencent à entrer dans l'estimation.

In [ ]:
grille_hill = np.arange(50, 1201, 50)

valeurs_hill = [  ## A completer ##
    ...  ## A completer ##
    for k in grille_hill
]

gamma_theorique = 1 / 2.5

plt.figure(figsize=(8, 4.5))
plt.plot(
    grille_hill,
    ...,  ## A completer ##
    linewidth=1.3,
    marker="o",
    markersize=3,
)
plt.axhline(
    gamma_theorique,
    linestyle="--",
    label="indice simulé 1/2.5",
)
plt.xlabel("Nombre d'extrêmes $k$")
plt.ylabel(r"$\widehat{\gamma}_k$")
plt.title("Graphe de Hill du rayon")
plt.legend()
plt.show()

k_hill = ...  ## A completer ##

print("Valeur retenue pour k_hill :", k_hill)

## Rappel de cours — Extrapolation d'un quantile extrême

Pour une probabilité de dépassement $p$ très petite, l'estimateur de Weissman est

$$
\widehat q(p)
=
X_{n-k:n}
\left(
\frac{k}{np}
\right)^{\widehat\gamma_k}.
$$

Cette formule combine une statistique d'ordre intermédiaire et l'extrapolation fournie par la variation régulière de la queue.

### Question 5

Estimer avec Weissman le quantile radial associé à `p_cible = 1e-5` en utilisant `k_hill`, puis calculer l'erreur relative par rapport à `quantile_rayon_reference`.

In [ ]:
p_cible = 1e-5
rayons_ordonnes = np.sort(
    donnees["R"].to_numpy()
)
n = len(rayons_ordonnes)

seuil_hill = ...  ## A completer ##
gamma_hill = ...  ## A completer ##

quantile_weissman = ...  ## A completer ##

erreur_relative = ...  ## A completer ##

print("Quantile estimé :", quantile_weissman)
print("Valeur de référence :", quantile_rayon_reference)
print("Erreur relative :", erreur_relative)

# Partie 2 — Classification dans les régions extrêmes

## Rappel de cours

Pour

$$
A_t=\{x:\lVert x\rVert>t\},
$$

le risque extrême est

$$
R_t(g)
=
\mathbb P\{g(X)\neq Y\mid X\in A_t\}.
$$

Le risque global peut donner très peu de poids à cette région lorsque $\mathbb P(X\in A_t)$ est faible. La taille d'échantillon utile est alors de l'ordre de

$$
k\simeq n\mathbb P(X\in A_t).
$$

Lorsque les deux classes ont une décroissance radiale comparable, le cours montre que le classifieur de Bayes limite devient angulaire. Il dépend de $\Theta(X)$ et devient constant le long d'un rayon. À seuil fini, le rayon peut encore contenir de l'information.

La comparaison utilise deux familles de modèles.

- `LinearSVC` construit une frontière de décision linéaire.
- `RandomForestClassifier` construit une frontière flexible à partir d'un ensemble d'arbres.

## Lien avec la minimisation du risque empirique

Le cours introduit la minimisation du risque empirique sous la forme

$$
\widehat g
\in
\operatorname*{arg\,min}_{g\in\mathcal G}
\widehat R_n(g),
\qquad
\widehat R_n(g)
=
\frac1n
\sum_{i=1}^n
\mathbf 1\{g(X_i)\neq Y_i\}.
$$

Le choix d'un algorithme de classification revient d'abord à choisir une **classe de fonctions** $\mathcal G$ dans laquelle la règle de décision sera cherchée.

Pour un classifieur linéaire sur $X\in\mathbb R^d$, la classe naturelle est

$$
\mathcal G_{\mathrm{lin}}
=
\left\{
g_{w,b}(x)
=
\mathbf 1\{w^\top x+b\geq 0\}
:
w\in\mathbb R^d,\ b\in\mathbb R
\right\}.
$$

Une minimisation idéale du risque empirique chercherait donc le meilleur hyperplan de séparation parmi tous les éléments de $\mathcal G_{\mathrm{lin}}$. En pratique, `LinearSVC` ne minimise pas directement la perte 0-1. Il minimise une perte hinge régularisée, plus facile à optimiser, tout en restant dans la même classe de frontières de décision linéaires.

Pour une forêt aléatoire, on peut d'abord définir $\mathcal T_D$ comme la classe des arbres de décision de profondeur au plus $D$, construits avec des coupures du type $x_j\leq s$. Pour un nombre d'arbres $B$, une classe de forêts peut s'écrire

$$
\mathcal G_{\mathrm{forêt}}(B,D)
=
\left\{
x
\mapsto
\mathbf 1
\left\{
\frac1B
\sum_{b=1}^{B}
T_b(x)
\geq
\frac12
\right\}
:
T_b\in\mathcal T_D
\right\}.
$$

Une minimisation empirique exacte sur cette classe serait très difficile. `RandomForestClassifier` construit chaque arbre de façon gloutonne à partir d'un échantillon bootstrap et de sous-ensembles aléatoires de variables, puis agrège les arbres par vote. Le modèle obtenu appartient donc à une classe de fonctions beaucoup plus riche que $\mathcal G_{\mathrm{lin}}$, même si l'algorithme ne calcule pas l'`argmin` exact du risque 0-1 sur toute cette classe.

Dans la région extrême, la même logique s'applique après restriction aux $k$ plus grands rayons. Pour une classe angulaire $\mathcal G_S$, le critère devient

$$
\widehat L_k(g)
=
\frac1k
\sum_{i=1}^{k}
\mathbf 1
\left\{
g\bigl(\Theta(X_{(i)})\bigr)
\neq
Y_{(i)}
\right\}.
$$

Le classifieur linéaire angulaire cherche alors dans une classe de séparateurs linéaires en fonction de $\Theta$, tandis que la forêt angulaire cherche dans une classe de votes d'arbres construits sur $\Theta$.

### Séparation entre apprentissage et test

Le jeu `test` sert uniquement à l'évaluation finale. Le choix de `k` est effectué à partir de `apprentissage`.

In [ ]:
apprentissage, test = train_test_split(
    donnees,
    test_size=0.30,
    stratify=donnees["Y"],
    random_state=42,
)

seuil_evaluation = np.quantile(
    apprentissage["R"],
    0.98,
)

print("Taille apprentissage :", len(apprentissage))
print("Taille test :", len(test))
print("Seuil d'évaluation :", seuil_evaluation)

## Rappel de cours — Estimer $R_t(g)$

Pour un seuil $t$,

$$
\widehat R_t(g)
=
\frac{
\sum_i
\mathbf 1\{g(X_i)\neq Y_i\}
\mathbf 1\{\lVert X_i\rVert>t\}
}{
\sum_i
\mathbf 1\{\lVert X_i\rVert>t\}
}.
$$

Plus la région est rare, plus le nombre d'observations disponibles pour estimer cette quantité est petit.

### Question 6

Compléter la fonction `risque_extreme`, puis afficher le nombre d'observations de `test` dont le rayon dépasse `seuil_evaluation`.

In [ ]:
def risque_extreme(
    y_vrai,
    predictions,
    rayons,
    seuil,
):
    y_vrai = np.asarray(y_vrai)
    predictions = np.asarray(predictions)
    rayons = np.asarray(rayons)

    masque = ...  ## A completer ##

    if masque.sum() == 0:
        return np.nan

    return ...  ## A completer ##


n_test_extreme = int(
    ...  ## A completer ##
)

print(
    "Nombre d'observations extrêmes dans le test :",
    n_test_extreme,
)

## Rappel — Classifieur global

Un classifieur global est entraîné sur toutes les observations d'apprentissage. Il vise donc d'abord une bonne performance moyenne sur la distribution globale. Rien ne garantit qu'il minimise `R_t(g)`.

Le classifieur linéaire est standardisé avec `StandardScaler`. La forêt aléatoire n'a pas besoin de cette standardisation.

### Question 7

Entraîner un classifieur linéaire global et une forêt aléatoire globale sur `(Z1, Z2)`, puis construire un tableau contenant le risque global et le risque extrême de chacun.

In [ ]:
lineaire_global = make_pipeline(
    StandardScaler(),
    LinearSVC(
        C=1.0,
        max_iter=20000,
        random_state=0,
    ),
)

foret_globale = RandomForestClassifier(
    n_estimators=120,
    max_depth=6,
    min_samples_leaf=8,
    max_features=1.0,
    n_jobs=1,
    random_state=0,
)

lineaire_global.fit(
    ...,  ## A completer ##
    ...,  ## A completer ##
)

foret_globale.fit(
    ...,  ## A completer ##
    ...,  ## A completer ##
)

pred_lineaire_global = ...  ## A completer ##
pred_foret_globale = ...  ## A completer ##

resultats_globaux = pd.DataFrame(
    {
        "modèle": [
            "linéaire global",
            "forêt globale",
        ],
        "risque global": [
            ...,  ## A completer ##
            ...,  ## A completer ##
        ],
        "risque extrême": [
            ...,  ## A completer ##
            ...,  ## A completer ##
        ],
    }
)

resultats_globaux

## Rappel de cours — Apprendre sur les $k$ plus grands rayons

Les rayons sont triés par ordre décroissant

$$
R_{(1)}\geq\cdots\geq R_{(n)}.
$$

Pour une famille de classifieurs angulaires, le cours considère

$$
\widehat L_k(g)
=
\frac1k
\sum_{i=1}^{k}
\mathbf 1\{g(\Theta(X_{(i)}))\neq Y_{(i)}\}.
$$

Le modèle est entraîné sur les $k$ observations les plus extrêmes et ne reçoit que l'angle en entrée.

### Question 8

Entraîner un classifieur linéaire angulaire sur les `k_hill` plus grands rayons de `apprentissage`, puis comparer son risque extrême à celui du classifieur linéaire global.

In [ ]:
indices_tries = np.argsort(
    apprentissage["R"].to_numpy()
)

indices_extremes_hill = ...  ## A completer ##
app_extreme_hill = ...  ## A completer ##

lineaire_angulaire_hill = make_pipeline(
    StandardScaler(),
    LinearSVC(
        C=1.0,
        max_iter=20000,
        random_state=0,
    ),
)

lineaire_angulaire_hill.fit(
    ...,  ## A completer ##
    ...,  ## A completer ##
)

pred_lineaire_angulaire_hill = ...  ## A completer ##

pd.DataFrame(
    {
        "modèle": [
            "linéaire global",
            "linéaire angulaire extrême",
        ],
        "risque extrême": [
            ...,  ## A completer ##
            ...,  ## A completer ##
        ],
    }
)

### Question 9

Entraîner une forêt aléatoire angulaire sur le même sous-échantillon `app_extreme_hill`, puis comparer sur la queue les quatre modèles déjà construits.

In [ ]:
foret_angulaire_hill = RandomForestClassifier(
    n_estimators=120,
    max_depth=6,
    min_samples_leaf=8,
    max_features=1.0,
    n_jobs=1,
    random_state=0,
)

foret_angulaire_hill.fit(
    ...,  ## A completer ##
    ...,  ## A completer ##
)

pred_foret_angulaire_hill = ...  ## A completer ##

predictions_modeles_hill = {
    "linéaire global": pred_lineaire_global,
    "forêt globale": pred_foret_globale,
    "linéaire angulaire extrême": pred_lineaire_angulaire_hill,
    "forêt angulaire extrême": pred_foret_angulaire_hill,
}

lignes = []

for nom, predictions in predictions_modeles_hill.items():
    lignes.append(
        {
            "modèle": nom,
            "risque extrême": ...  ## A completer ##
        }
    )

comparaison_hill = pd.DataFrame(lignes)

comparaison_hill.sort_values(
    "risque extrême"
)

### Question 10

L'objectif est d'illustrer la différence entre un classifieur global qui utilise les coordonnées $(Z_1,Z_2)$ et un classifieur angulaire qui ne reçoit que $\Theta_1$.

Pour un rayon fixé $R=r$ et une valeur de $\Theta_1$, les coordonnées sont

$$
Z_1=r\Theta_1,
\qquad
Z_2=r(1-\Theta_1).
$$

Avec `LinearSVC`, la fonction `decision_function` renvoie un **score de décision**. Ce score est une quantité signée utilisée pour choisir la classe. Un score positif conduit à la classe 1, un score négatif à la classe 0 et la valeur 0 correspond à la frontière de décision. Ce score n'est pas une probabilité.

Tracer le score de décision du classifieur linéaire global en fonction de `Theta1` pour les rayons 4, 8 et 16, puis superposer le score du classifieur linéaire angulaire. Le graphique doit permettre de voir si la décision peut changer avec le rayon alors que l'angle reste fixé.

In [ ]:
grille_theta = np.linspace(
    0.05,
    0.95,
    200,
)

plt.figure(figsize=(8, 5))

for rayon in [4, 8, 16]:
    grille_z = pd.DataFrame(
        {
            "Z1": ...,  ## A completer ##
            "Z2": ...,  ## A completer ##
        }
    )

    score = ...  ## A completer ##

    plt.plot(
        grille_theta,
        score,
        linewidth=1.5,
        label=f"linéaire global, R={rayon}",
    )

grille_angle = pd.DataFrame(
    {
        "Theta1": ...  ## A completer ##
    }
)

score_angulaire = ...  ## A completer ##

plt.plot(
    grille_theta,
    score_angulaire,
    linewidth=3,
    label="linéaire angulaire extrême",
)

plt.axhline(
    0,
    linestyle="--",
)
plt.xlabel(r"$\Theta_1$")
plt.ylabel("Score de décision")
plt.title("Dépendance au rayon et à l'angle")
plt.legend()
plt.show()

## Rappel de cours — Choisir $k$ pour la classification

Le régime intermédiaire impose

$$
k_n\to\infty,
\qquad
\frac{k_n}{n}\to0.
$$

À taille finie, un $k$ trop petit donne peu de données pour apprendre. Un $k$ trop grand fait entrer des observations du corps et peut dégrader le modèle extrême.

Le graphe de Hill choisit $k$ pour stabiliser l'estimation de la queue radiale. Pour la classification, un autre choix naturel consiste à sélectionner $k$ en minimisant une estimation du risque extrême par validation croisée.

Sur les données simulées, le signal angulaire de la queue est non linéaire. La validation croisée est donc effectuée avec une forêt aléatoire angulaire.

### Question 11

Compléter `validation_croisee_extreme`, l'appliquer à `grille_k`, puis choisir `k_cv` comme la valeur qui minimise le risque extrême moyen. La courbe doit être interprétée comme un compromis entre variance pour les petits `k` et contamination par le corps pour les grands `k`.

In [ ]:
def validation_croisee_extreme(
    donnees_app,
    grille_k,
    quantile_cible=0.98,
    n_plis=5,
):
    cv = StratifiedKFold(
        n_splits=n_plis,
        shuffle=True,
        random_state=7,
    )

    y = donnees_app["Y"].to_numpy()
    n_total = len(donnees_app)
    resultats = []

    for k in grille_k:
        risques = []
        proportion_k = ...  ## A completer ##

        for indices_app, indices_val in cv.split(
            np.zeros(n_total),
            y,
        ):
            pli_app = ...  ## A completer ##
            pli_val = ...  ## A completer ##

            k_pli = ...  ## A completer ##

            indices_tries = ...  ## A completer ##
            pli_extreme = ...  ## A completer ##

            modele = RandomForestClassifier(
                n_estimators=80,
                max_depth=6,
                min_samples_leaf=8,
                max_features=1.0,
                n_jobs=1,
                random_state=0,
            )

            modele.fit(
                ...,  ## A completer ##
                ...,  ## A completer ##
            )

            seuil_pli = ...  ## A completer ##
            predictions = ...  ## A completer ##

            risque = ...  ## A completer ##
            risques.append(risque)

        resultats.append(
            {
                "k": k,
                "risque_moyen": np.nanmean(risques),
                "ecart_type": np.nanstd(risques),
            }
        )

    return pd.DataFrame(resultats)


grille_k = [
    60,
    100,
    150,
    200,
    250,
    300,
    350,
    400,
    450,
    500,
    600,
    800,
]

table_cv = ...  ## A completer ##
k_cv = ...  ## A completer ##

plt.figure(figsize=(8, 4.5))
plt.errorbar(
    table_cv["k"],
    table_cv["risque_moyen"],
    yerr=table_cv["ecart_type"],
    marker="o",
    capsize=3,
    linewidth=1.3,
)
plt.axvline(
    k_cv,
    linestyle="--",
    label=f"k_cv={k_cv}",
)
plt.xlabel("Nombre d'extrêmes $k$")
plt.ylabel("Risque extrême en validation croisée")
plt.title("Choix de $k$ pour la classification")
plt.legend()
plt.show()

table_cv

## Point méthodologique — Quand calculer le seuil en validation croisée ?

Deux procédures peuvent sembler naturelles.

**Procédure A.** Le seuil est calculé une seule fois sur tout `apprentissage`. Les observations situées sous ce seuil sont retirées, puis la validation croisée est effectuée uniquement sur les observations restantes.

**Procédure B.** Les plis sont construits d'abord. Dans chaque pli, le seuil et les `k_pli` observations extrêmes sont déterminés uniquement à partir du bloc d'apprentissage du pli. Le bloc de validation est ensuite évalué avec ce seuil.

### Question 12

L'objectif de la validation croisée est d'estimer la performance de la **procédure complète** qui sera appliquée à un nouvel échantillon d'apprentissage, c'est-à-dire choisir un seuil à partir des données, sélectionner les observations extrêmes, entraîner le classifieur puis prédire sur de nouvelles observations.

Comparer les procédures A et B en indiquant laquelle reproduit ce scénario sans utiliser les observations du bloc de validation pour déterminer le seuil. Expliquer pourquoi l'autre procédure crée une fuite d'information et dans quel sens cette fuite peut rendre l'estimation du risque trop optimiste.

**Réponse :**

### Question 13

Réentraîner une forêt aléatoire angulaire avec `k_cv`, puis comparer dans un même tableau le risque global et le risque extrême de la forêt globale, de la forêt angulaire choisie avec `k_hill` et de la forêt angulaire choisie avec `k_cv`.

In [ ]:
indices_tries = np.argsort(
    apprentissage["R"].to_numpy()
)

indices_extremes_cv = ...  ## A completer ##
app_extreme_cv = ...  ## A completer ##

foret_angulaire_cv = RandomForestClassifier(
    n_estimators=120,
    max_depth=6,
    min_samples_leaf=8,
    max_features=1.0,
    n_jobs=1,
    random_state=0,
)

foret_angulaire_cv.fit(
    ...,  ## A completer ##
    ...,  ## A completer ##
)

pred_foret_angulaire_cv = ...  ## A completer ##

predictions_finales = {
    "forêt globale": pred_foret_globale,
    "forêt angulaire Hill": pred_foret_angulaire_hill,
    "forêt angulaire CV": pred_foret_angulaire_cv,
}

lignes = []

for nom, predictions in predictions_finales.items():
    lignes.append(
        {
            "modèle": nom,
            "risque global": ...,  ## A completer ##
            "risque extrême": ...,  ## A completer ##
        }
    )

comparaison_finale = pd.DataFrame(lignes)

comparaison_finale

### Question 14 — Ablation : l'angle suffit-il ?

La forêt angulaire construite avec `k_cv` diffère de la forêt globale sur deux points à la fois. Elle utilise seulement `Theta1` et elle est entraînée uniquement sur les observations extrêmes.

Pour isoler l'effet du seuillage, entraîner une nouvelle forêt `foret_angulaire_toutes` qui utilise seulement `Theta1`, mais qui est entraînée sur **toutes** les observations de `apprentissage`. Comparer ensuite son risque extrême à celui de `foret_globale` et de `foret_angulaire_cv`.

L'objectif est de déterminer si le gain observé vient seulement du changement de représentation $(Z_1,Z_2)\mapsto\Theta_1$, ou s'il vient aussi du fait de cibler l'apprentissage sur la région extrême.

In [ ]:
foret_angulaire_toutes = RandomForestClassifier(
    n_estimators=120,
    max_depth=6,
    min_samples_leaf=8,
    max_features=1.0,
    n_jobs=1,
    random_state=0,
)

foret_angulaire_toutes.fit(
    ...,  ## A completer ##
    ...,  ## A completer ##
)

pred_foret_angulaire_toutes = ...  ## A completer ##

comparaison_ablation = pd.DataFrame(
    {
        "modèle": [
            "forêt globale",
            "forêt angulaire toutes données",
            "forêt angulaire extrême CV",
        ],
        "risque extrême": [
            ...,  ## A completer ##
            ...,  ## A completer ##
            ...,  ## A completer ##
        ],
    }
)

comparaison_ablation

# Partie 3 — Un classifieur angulaire sur des données réelles

## Données Breast Cancer Wisconsin Diagnostic

Le jeu *Breast Cancer Wisconsin Diagnostic* contient 569 observations et 30 variables quantitatives décrivant des propriétés morphologiques calculées à partir d'images de noyaux cellulaires. L'étiquette indique si la tumeur est maligne ou bénigne.

Une région extrême est intéressante ici pour une raison méthodologique. Une observation dont le rayon est très grand après standardisation marginale correspond à une tumeur dont plusieurs caractéristiques sont atypiques par rapport à l'échantillon d'apprentissage. Ces cas sont peu nombreux. Une bonne performance moyenne peut donc masquer une performance différente sur ces observations atypiques.

Cette partie ne cherche pas à établir une règle médicale sur les tumeurs extrêmes. Le jeu sert à examiner si la démarche du cours reste pertinente sur des données réelles en dimension 30, où la structure de queue n'est pas contrôlée par construction.

La version utilisée est fournie directement par `scikit-learn` avec `load_breast_cancer`.

Les données originales peuvent aussi être obtenues sur la page UCI suivante :

<https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic>

### Question 15

Charger les données avec `load_breast_cancer`, coder `Y_reel = 1` pour une tumeur maligne, puis afficher la dimension de `X_reel` et la proportion de la classe 1.

In [ ]:
cancer = load_breast_cancer(
    as_frame=True
)

X_reel = ...  ## A completer ##
Y_reel = ...  ## A completer ##

print("Dimension :", X_reel.shape)

print(
    "Proportion de tumeurs malignes :",
    Y_reel.mean(),
)

## Rappel de cours — Mettre les marges sur une échelle commune

Les 30 variables n'ont pas les mêmes unités. Une variable mesurée sur une grande échelle pourrait dominer artificiellement le rayon.

Pour une marge continue $F_j$, le cours propose la transformation idéale

$$
Z_j
=
\frac{1}{1-F_j(X_j)}.
$$

En pratique, $F_j$ est remplacée ici par sa fonction de répartition empirique calculée sur le bloc d'apprentissage.

### Question 16

Compléter `transformation_pareto_empirique`, séparer les données en apprentissage et test, appliquer la transformation, puis construire `R_reel_app`, `R_reel_test`, `Theta_reel_app` et `Theta_reel_test` avec la norme $\ell_1$.

In [ ]:
def transformation_pareto_empirique(
    X_app,
    X_test,
):
    Z_app = pd.DataFrame(
        index=X_app.index,
        columns=X_app.columns,
        dtype=float,
    )

    Z_test = pd.DataFrame(
        index=X_test.index,
        columns=X_test.columns,
        dtype=float,
    )

    n = len(X_app)

    for colonne in X_app.columns:
        valeurs_app = (
            X_app[colonne].to_numpy()
        )

        rangs_app = ...  ## A completer ##

        Z_app[colonne] = ...  ## A completer ##

        valeurs_triees = ...  ## A completer ##

        rangs_test = ...  ## A completer ##

        Z_test[colonne] = ...  ## A completer ##

    return Z_app, Z_test


(
    X_reel_app,
    X_reel_test,
    Y_reel_app,
    Y_reel_test,
) = train_test_split(
    X_reel,
    Y_reel,
    test_size=0.30,
    stratify=Y_reel,
    random_state=42,
)

Z_reel_app, Z_reel_test = ...  ## A completer ##

R_reel_app = ...  ## A completer ##
R_reel_test = ...  ## A completer ##

Theta_reel_app = ...  ## A completer ##
Theta_reel_test = ...  ## A completer ##

print(
    "Rayon médian apprentissage :",
    np.median(R_reel_app),
)

print(
    "Rayon maximal apprentissage :",
    np.max(R_reel_app),
)

## Choix provisoire d'une région extrême

Le jeu réel est petit. Un seuil très haut laisserait trop peu d'observations pour entraîner et évaluer un modèle.

Un premier choix fixe donc la région extrême aux 20 % plus grands rayons du bloc d'apprentissage. Cette valeur n'est pas une recommandation générale. Elle sert de point de départ avant de discuter le choix de $k$ dans la question suivante.

Pour comparer les modèles, deux quantités sont utilisées.

- Le **risque global empirique** est la proportion d'erreurs sur tout le bloc de test.
- Le **risque extrême empirique** est la proportion d'erreurs uniquement parmi les observations du test dont le rayon dépasse le seuil calculé sur l'apprentissage.

Dans les deux colonnes, une valeur plus petite est meilleure.

### Question 17

En prenant `k_reel` égal à 20 % de la taille du bloc d'apprentissage, entraîner un classifieur linéaire global, une forêt globale, un classifieur linéaire angulaire extrême et une forêt angulaire extrême. Construire ensuite un tableau avec le risque global et le risque extrême de chaque modèle en utilisant le même `seuil_reel` pour les quatre modèles.

In [ ]:
proportion_extreme_reelle = 0.20

k_reel = ...  ## A completer ##

indices_reels = ...  ## A completer ##
indices_reels_extremes = ...  ## A completer ##

seuil_reel = ...  ## A completer ##

lineaire_reel_global = make_pipeline(
    StandardScaler(),
    LinearSVC(
        C=1.0,
        max_iter=30000,
        random_state=0,
    ),
)

foret_reelle_globale = RandomForestClassifier(
    n_estimators=120,
    min_samples_leaf=4,
    n_jobs=1,
    random_state=0,
)

lineaire_reel_angulaire = make_pipeline(
    StandardScaler(),
    LinearSVC(
        C=1.0,
        max_iter=30000,
        random_state=0,
    ),
)

foret_reelle_angulaire = RandomForestClassifier(
    n_estimators=120,
    min_samples_leaf=3,
    n_jobs=1,
    random_state=0,
)

lineaire_reel_global.fit(
    ...,  ## A completer ##
    ...,  ## A completer ##
)

foret_reelle_globale.fit(
    ...,  ## A completer ##
    ...,  ## A completer ##
)

lineaire_reel_angulaire.fit(
    ...,  ## A completer ##
    ...,  ## A completer ##
)

foret_reelle_angulaire.fit(
    ...,  ## A completer ##
    ...,  ## A completer ##
)

predictions_reelles = {
    "linéaire global": ...,  ## A completer ##
    "forêt globale": ...,  ## A completer ##
    "linéaire angulaire extrême": ...,  ## A completer ##
    "forêt angulaire extrême": ...,  ## A completer ##
}

lignes = []

for nom, predictions in predictions_reelles.items():
    lignes.append(
        {
            "modèle": nom,
            "risque global": ...,  ## A completer ##
            "risque extrême": ...,  ## A completer ##
        }
    )

comparaison_reelle = pd.DataFrame(
    lignes
)

print(
    "Nombre de points du test dans la région extrême :",
    int(
        (
            R_reel_test > seuil_reel
        ).sum()
    ),
)

comparaison_reelle.sort_values(
    "risque extrême"
)

## Revenir sur le choix des 20 % extrêmes

Le choix `proportion_extreme_reelle = 0.20` est pratique mais arbitraire.

Deux outils du TP permettent de le discuter.

Le graphe de Hill examine la stabilité de l'estimation de la queue radiale. La validation croisée extrême choisit directement $k$ pour la performance de classification dans une région rare.

Sur un petit jeu réel, ces outils peuvent être beaucoup moins stables que sur les données simulées. Cette instabilité est elle-même une information : elle signale qu'il faut être prudent avant de présenter une valeur de $k$ comme évidente.

Pour la validation croisée, la transformation de Pareto, le rayon, les angles et le seuil sont recalculés uniquement à partir du bloc d'apprentissage de chaque pli.

In [ ]:
def validation_croisee_angulaire_reelle(
    X_app,
    y_app,
    grille_k,
    quantile_cible=0.80,
    n_plis=5,
):
    cv = StratifiedKFold(
        n_splits=n_plis,
        shuffle=True,
        random_state=17,
    )

    n_total = len(X_app)
    resultats = []

    for k in grille_k:
        risques = []
        proportion_k = k / n_total

        for indices_app, indices_val in cv.split(
            X_app,
            y_app,
        ):
            X_pli_app = X_app.iloc[
                indices_app
            ]
            X_pli_val = X_app.iloc[
                indices_val
            ]
            y_pli_app = y_app.iloc[
                indices_app
            ]
            y_pli_val = y_app.iloc[
                indices_val
            ]

            (
                Z_pli_app,
                Z_pli_val,
            ) = transformation_pareto_empirique(
                X_pli_app,
                X_pli_val,
            )

            R_pli_app = Z_pli_app.sum(axis=1)
            R_pli_val = Z_pli_val.sum(axis=1)

            Theta_pli_app = Z_pli_app.div(
                R_pli_app,
                axis=0,
            )

            Theta_pli_val = Z_pli_val.div(
                R_pli_val,
                axis=0,
            )

            k_pli = max(
                12,
                int(
                    round(
                        proportion_k
                        * len(X_pli_app)
                    )
                ),
            )

            indices_tries = np.argsort(
                R_pli_app.to_numpy()
            )

            indices_extremes = indices_tries[
                -k_pli:
            ]

            modele = RandomForestClassifier(
                n_estimators=80,
                min_samples_leaf=3,
                n_jobs=1,
                random_state=0,
            )

            modele.fit(
                Theta_pli_app.iloc[
                    indices_extremes
                ],
                y_pli_app.iloc[
                    indices_extremes
                ],
            )

            seuil_pli = np.quantile(
                R_pli_app,
                quantile_cible,
            )

            predictions = modele.predict(
                Theta_pli_val
            )

            risques.append(
                risque_extreme(
                    y_pli_val,
                    predictions,
                    R_pli_val,
                    seuil_pli,
                )
            )

        resultats.append(
            {
                "k": k,
                "risque_moyen": np.nanmean(
                    risques
                ),
                "ecart_type": np.nanstd(
                    risques
                ),
            }
        )

    return pd.DataFrame(resultats)

### Question 18

Critiquer le choix des 20 % plus grands rayons en traçant le graphe de Hill de `R_reel_app` et le risque extrême de validation croisée pour plusieurs valeurs de `k`. Comparer `k_reel` à la zone de stabilité éventuelle du graphe de Hill et à la valeur `k_reel_cv` minimisant le risque moyen. Conclure en tenant compte de la petite taille du jeu réel.

In [ ]:
grille_hill_reel = np.arange(
    10,
    181,
    5,
)

valeurs_hill_reel = [  ## A completer ##
    ...  ## A completer ##
    for k in grille_hill_reel
]

plt.figure(figsize=(8, 4.5))
plt.plot(
    grille_hill_reel,
    ...,  ## A completer ##
    marker="o",
    markersize=3,
    linewidth=1.2,
)
plt.axvline(
    k_reel,
    linestyle="--",
    label=f"20 % des données : k={k_reel}",
)
plt.xlabel("Nombre d'extrêmes $k$")
plt.ylabel(r"$\widehat{\gamma}_k$")
plt.title("Graphe de Hill du rayon sur les données réelles")
plt.legend()
plt.show()

grille_k_reel = [
    30,
    40,
    50,
    60,
    70,
    80,
    90,
    100,
    110,
    120,
    130,
]

table_cv_reel = ...  ## A completer ##
k_reel_cv = ...  ## A completer ##

plt.figure(figsize=(8, 4.5))
plt.errorbar(
    ...,  ## A completer ##
    ...,  ## A completer ##
    yerr=...,  ## A completer ##
    marker="o",
    capsize=3,
    linewidth=1.2,
)
plt.axvline(
    k_reel,
    linestyle=":",
    label=f"choix 20 % : k={k_reel}",
)
plt.axvline(
    k_reel_cv,
    linestyle="--",
    label=f"choix CV : k={k_reel_cv}",
)
plt.xlabel("Nombre d'extrêmes $k$")
plt.ylabel("Risque extrême en validation croisée")
plt.title("Choix de $k$ sur les données réelles")
plt.legend()
plt.show()

print(
    "k correspondant à 20 % :",
    k_reel,
)

print(
    "k choisi par validation croisée :",
    k_reel_cv,
)

table_cv_reel

# Aide Python

### `np.sort`

`np.sort(x)` renvoie les valeurs de `x` triées par ordre croissant.

```python
x_trie = np.sort(x)
```

### `np.argsort`

`np.argsort(x)` renvoie les indices qui permettraient de trier `x`.

```python
indices = np.argsort(x)
```

Les indices des `k` plus grandes valeurs sont obtenus par

```python
indices_extremes = np.argsort(x)[-k:]
```

### `np.quantile`

`np.quantile(x, 0.97)` calcule le quantile empirique d'ordre 0.97.

```python
seuil = np.quantile(x, 0.97)
```

### `train_test_split`

`train_test_split` sépare les données en blocs d'apprentissage et de test. `stratify` conserve approximativement la proportion des classes.

```python
app, test = train_test_split(
    donnees,
    test_size=0.30,
    stratify=donnees["Y"],
    random_state=42,
)
```

### `LinearSVC`

`LinearSVC` construit un classifieur à frontière linéaire.

```python
modele = make_pipeline(
    StandardScaler(),
    LinearSVC(C=1.0),
)

modele.fit(X_app, y_app)
predictions = modele.predict(X_test)
```

`decision_function` renvoie un score signé. Un score positif correspond au côté de la frontière associé à la classe 1.

```python
scores = modele.decision_function(
    X_test
)
```

### `RandomForestClassifier`

`RandomForestClassifier` construit plusieurs arbres et agrège leurs prédictions.

```python
modele = RandomForestClassifier(
    n_estimators=200,
    random_state=0,
)

modele.fit(X_app, y_app)
predictions = modele.predict(X_test)
```

### `StratifiedKFold`

`StratifiedKFold` construit des plis de validation croisée en conservant approximativement la proportion des classes.

```python
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=0,
)
```

### `pd.Series.rank`

`pd.Series.rank` renvoie les rangs empiriques des observations.

```python
rangs = (
    pd.Series(valeurs)
    .rank(method="average")
    .to_numpy()
)
```

### `np.searchsorted`

`np.searchsorted` permet de positionner une valeur dans un tableau trié.

```python
rangs_test = np.searchsorted(
    valeurs_triees,
    nouvelles_valeurs,
    side="right",
)
```

### `load_breast_cancer`

`load_breast_cancer` charge le jeu Breast Cancer Wisconsin Diagnostic fourni avec `scikit-learn`.

```python
cancer = load_breast_cancer(
    as_frame=True
)
```